# Day 10: Custom DAG Pipeline Orchestrator with Retries & Fault Tolerance
> **Part 10 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2010%20of%2010-008080?style=for-the-badge)](../README.md)

---

## 🎯 Day 10: Custom DAG Pipeline Orchestrator with Retries & Fault Tolerance
### 🎯 Learning Objectives:
1. Build an Airflow-style **Directed Acyclic Graph (DAG) Pipeline Orchestrator** from scratch in Python (`SimpleOrchestrator`).
2. Implement core workflow management capabilities:
   - Task registration and sequential dependency resolution.
   - Shared execution context dictionary for inter-task data passing.
   - Automatic retries with customizable retry counts and delay backoff.
   - Comprehensive task state machine: `PENDING` -> `RUNNING` -> `SUCCESS` / `FAILED`.
3. Conduct **Fault Injection & Chaos Testing**:
   - Verify that transient errors trigger retries.
   - Verify that persistent failures halt downstream execution safely.
4. Understand architectural boundaries: PySpark as the **Compute Engine** vs. Airflow/DAGs as the **Orchestration Layer**.
---


In [297]:
#DAY 10
# Step 1: Define Your Pipeline Tasks (Cell 1)
import logging
import time

# Configure logging to mimic Airflow's task logs
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def extract():
    logging.info("--- TASK START: Extract ---")
    logging.info("Reading data from Neon PostgreSQL (or CSV)...")
    # Simulate data extraction
    data = [{"id": 1, "item": "Laptop", "price": 1000}, {"id": 2, "item": "Mouse", "price": 50}]
    logging.info("Extract completed successfully.")
    return data

def validate(data):
    logging.info("--- TASK START: Validate ---")
    if not data:
        raise ValueError("Validation failed: Dataset is empty!")
    logging.info(f"Validated {len(data)} records successfully.")
    return data

def transform(data):
    logging.info("--- TASK START: Transform ---")
    # Let's add total revenue calculation
    for row in data:
        row["total_value"] = row["price"] * 1.18  # adding tax
    logging.info("Transformation completed successfully.")
    return data

def load(data):
    logging.info("--- TASK START: Load ---")
    logging.info(f"Loading {len(data)} records into target destination...")
    logging.info("Load completed successfully.")

def quality_check():
    logging.info("--- TASK START: Quality Check ---")
    logging.info("Running post-load data checks... All constraints passed!")

### 🧠 Custom DAG Orchestrator Engine Class (`SimpleOrchestrator`)


In [298]:
# Step 2: Build the Orchestrator Engine with Retries & Failure Handling (Cell 2)
class SimpleOrchestrator:
    def __init__(self, dag_name, max_retries=2):
        self.dag_name = dag_name
        self.max_retries = max_retries
        self.tasks = []

    def add_task(self, name, func, *args):
        self.tasks.append({"name": name, "func": func, "args": args})

    def run(self):
        logging.info(f"=== PIPELINE START: {self.dag_name} ===")
        context_data = None

        for task in self.tasks:
            task_name = task["name"]
            func = task["func"]
            args = task["args"]
            
            success = False
            attempt = 0

            while attempt <= self.max_retries and not success:
                try:
                    if attempt > 0:
                        logging.warning(f"RETRY {attempt}/{self.max_retries} for task: {task_name}")
                    
                    # Execute task (passing context data if it exists from previous task)
                    if args:
                        context_data = func(*args)
                    elif context_data is not None:
                        context_data = func(context_data)
                    else:
                        context_data = func()
                        
                    success = True
                    logging.info(f"Task '{task_name}' status: SUCCESS ✅")
                    
                except Exception as e:
                    attempt += 1
                    logging.error(f"Task '{task_name}' failed with error: {e}")
                    if attempt > self.max_retries:
                        logging.critical(f"Task '{task_name}' exceeded max retries. Pipeline HALTED ❌")
                        logging.info(f"=== PIPELINE FAILED: {self.dag_name} ===")
                        return False
                        
        logging.info(f"=== PIPELINE SUCCESS: {self.dag_name} ===")
        return True

### 🧠 Custom DAG Orchestrator Engine Class (`SimpleOrchestrator`)


In [299]:
# Step 3: Run and Test Your Pipeline (Cell 3)
# Initialize your pipeline DAG
pipeline = SimpleOrchestrator(dag_name="day10_ecommerce_pipeline", max_retries=2)

# Register tasks in order (Dependencies)
pipeline.add_task("extract", extract)
pipeline.add_task("validate", validate) # takes output of extract automatically
pipeline.add_task("transform", transform) # takes output of validate automatically
pipeline.add_task("load", load)
pipeline.add_task("quality_check", quality_check)

# Execute the orchestrated pipeline
pipeline.run()

2026-10-08 07:45:19,511 - INFO - === PIPELINE START: day10_ecommerce_pipeline ===
2026-10-08 07:45:19,512 - INFO - --- TASK START: Extract ---
2026-10-08 07:45:19,513 - INFO - Reading data from Neon PostgreSQL (or CSV)...
2026-10-08 07:45:19,514 - INFO - Extract completed successfully.
2026-10-08 07:45:19,515 - INFO - Task 'extract' status: SUCCESS ✅
2026-10-08 07:45:19,515 - INFO - --- TASK START: Validate ---
2026-10-08 07:45:19,516 - INFO - Validated 2 records successfully.
2026-10-08 07:45:19,516 - INFO - Task 'validate' status: SUCCESS ✅
2026-10-08 07:45:19,517 - INFO - --- TASK START: Transform ---
2026-10-08 07:45:19,518 - INFO - Transformation completed successfully.
2026-10-08 07:45:19,519 - INFO - Task 'transform' status: SUCCESS ✅
2026-10-08 07:45:19,519 - INFO - --- TASK START: Load ---
2026-10-08 07:45:19,520 - INFO - Loading 2 records into target destination...
2026-10-08 07:45:19,521 - INFO - Load completed successfully.
2026-10-08 07:45:19,521 - INFO - Task 'load' statu

True

### 🧠 Custom DAG Orchestrator Engine Class (`SimpleOrchestrator`)


In [300]:
# 🧪 Test the Failure & Retry Logic (Cell 4)
# Let's create a broken transform function to test retries and failure halt
def broken_transform(data):
    logging.info("--- TASK START: Transform (Broken) ---")
    raise Exception("Database connection timeout during transformation!")

# Build a failing pipeline
failing_pipeline = SimpleOrchestrator(dag_name="day10_failing_pipeline", max_retries=2)

failing_pipeline.add_task("extract", extract)
failing_pipeline.add_task("validate", validate)
failing_pipeline.add_task("transform", broken_transform) # This will fail and retry 2 times
failing_pipeline.add_task("load", load) # Should be skipped because transform fails

failing_pipeline.run()

2026-10-08 07:45:43,519 - INFO - === PIPELINE START: day10_failing_pipeline ===
2026-10-08 07:45:43,520 - INFO - --- TASK START: Extract ---
2026-10-08 07:45:43,520 - INFO - Reading data from Neon PostgreSQL (or CSV)...
2026-10-08 07:45:43,522 - INFO - Extract completed successfully.
2026-10-08 07:45:43,522 - INFO - Task 'extract' status: SUCCESS ✅
2026-10-08 07:45:43,524 - INFO - --- TASK START: Validate ---
2026-10-08 07:45:43,525 - INFO - Validated 2 records successfully.
2026-10-08 07:45:43,525 - INFO - Task 'validate' status: SUCCESS ✅
2026-10-08 07:45:43,526 - INFO - --- TASK START: Transform (Broken) ---
2026-10-08 07:45:43,526 - ERROR - Task 'transform' failed with error: Database connection timeout during transformation!
2026-10-08 07:45:43,527 - WARNING - RETRY 1/2 for task: transform
2026-10-08 07:45:43,528 - INFO - --- TASK START: Transform (Broken) ---
2026-10-08 07:45:43,529 - ERROR - Task 'transform' failed with error: Database connection timeout during transformation!
2

False

PySpark
   ↓
Processes/transforms data

Airflow
   ↓
Orchestrates when/how pipeline tasks run


---
### 🏆 Congratulations! You Completed the 10-Day PySpark Masterclass!
You have successfully built:
1. Medallion Lakehouse Ingestion (Bronze, Silver, Gold)
2. Production Data Quality & Quarantine Architecture
3. Incremental Change Ingestion & Metadata State Auditing
4. High-Performance Partitioning, Pruning & Broadcast Joins
5. Multi-Table Relational Modeling & Clean Star Schema Joins
6. Advanced Window Functions & Financial Time-Series Analytics
7. Reusable Modular Software Engineering for Big Data ETL
8. Enterprise Logging, Configuration & Automated Unit Testing
9. Serverless PostgreSQL Cloud Ingestion into a Lakehouse
10. A Fault-Tolerant DAG Pipeline Orchestrator with Retries

⭐ *Don't forget to star the repository and share your project!*
